# Phân loại bình luận độc hại tiếng Việt

## Thư viện

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import re
from datetime import datetime

import numpy as np
import pandas as pd

## Cấu hình

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        has_vihsd = (p / "data" / "processed" / "test_processed.csv").exists()
        has_vihos = (p / "data" / "vihos" / "repo" / "data").exists()
        if has_vihsd and has_vihos:
            return p
    raise FileNotFoundError(
        "Khong tim thay project root. Hay dat notebook trong thu muc notebooks/ "
        "hoac chay tai root repo co data/processed va data/vihos/repo/data."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VIHOS_DATA_DIR = DATA_DIR / "vihos" / "repo" / "data"

SPAN_DIR = VIHOS_DATA_DIR / "Span_Extraction_based_version"
VIHOS_TEST_DIR = VIHOS_DATA_DIR / "Test_data"
BIO_DIR = VIHOS_DATA_DIR / "Sequence_labeling_based_version"

RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESOURCES_DIR = PROJECT_ROOT / "outputs" / "resources"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESOURCES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VIHOS_DATA_DIR:", VIHOS_DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESOURCES_DIR:", RESOURCES_DIR)

PROJECT_ROOT: d:\Thư bae ngốc ngếch
VIHOS_DATA_DIR: d:\Thư bae ngốc ngếch\data\vihos\repo\data
RESULTS_DIR: d:\Thư bae ngốc ngếch\outputs\results
RESOURCES_DIR: d:\Thư bae ngốc ngếch\outputs\resources


## Hàm dùng chung

In [3]:
LABEL_MAP = {
    0: "CLEAN",
    1: "OFFENSIVE",
    2: "HATE",
}


def read_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Khong thay file: {path}")
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="utf-8-sig")


def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def text_hash(text):
    return hashlib.md5(normalize_text(text).encode("utf-8")).hexdigest()


def rel_path(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def parse_index_spans(value):
    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()
        if value in ["", "[]", "nan", "None"]:
            return []
        try:
            value = ast.literal_eval(value)
        except Exception:
            return []

    if isinstance(value, (int, np.integer)):
        return [int(value)]

    if not isinstance(value, (list, tuple, set)):
        return []

    out = []
    for item in value:
        if isinstance(item, (list, tuple, set)):
            out.extend(parse_index_spans(list(item)))
        else:
            try:
                out.append(int(item))
            except Exception:
                pass
    return sorted(set(out))


def indices_to_ranges(indices, text_len):
    indices = sorted(i for i in set(indices) if 0 <= i < text_len)
    if not indices:
        return []

    ranges = []
    start = prev = indices[0]
    for idx in indices[1:]:
        if idx == prev + 1:
            prev = idx
        else:
            ranges.append([start, prev + 1])
            start = prev = idx
    ranges.append([start, prev + 1])
    return ranges


def get_span_texts(text, ranges):
    spans = []
    for start, end in ranges:
        span = str(text)[start:end].strip()
        if span:
            spans.append(span)
    return spans

## ViHSD test

In [4]:
vihsd_test_path = PROCESSED_DIR / "test_processed.csv"
vihsd_test = read_csv(vihsd_test_path)

text_col = "text_clean" if "text_clean" in vihsd_test.columns else "free_text"
label_col = "label_id"

if text_col not in vihsd_test.columns:
    raise ValueError(f"Khong thay cot text trong {vihsd_test_path}")
if label_col not in vihsd_test.columns:
    raise ValueError(f"Khong thay cot label_id trong {vihsd_test_path}")

vihsd_test_std = pd.DataFrame({
    "sample_id": [f"vihsd_test_{i}" for i in range(len(vihsd_test))],
    "text": vihsd_test[text_col].fillna("").astype(str),
    "label_id": vihsd_test[label_col].astype(int),
})

vihsd_test_std["text_norm"] = vihsd_test_std["text"].map(normalize_text)
vihsd_test_std["label_name"] = vihsd_test_std["label_id"].map(LABEL_MAP)
vihsd_test_std["source_file"] = rel_path(vihsd_test_path)
vihsd_test_std["text_hash"] = vihsd_test_std["text_norm"].map(text_hash)

vihsd_test_std = vihsd_test_std[[
    "sample_id", "text", "text_norm", "label_id", "label_name", "source_file", "text_hash"
]]

vihsd_test_out = RESOURCES_DIR / "vihsd_test_clean_standard.csv"
vihsd_test_std.to_csv(vihsd_test_out, index=False)

vihsd_test_std.head()

,sample_id,text,text_norm,label_id,label_name,source_file,text_hash
0,vihsd_test_0,Đừng cố biện minh =)))) choi lon,đừng cố biện minh =)))) choi lon,0,CLEAN,data\processed\test_processed.csv,d9c5c51de8779369898398124a583e2c
1,vihsd_test_1,Haizz. Nthe này thì dân khổ quá,haizz. nthe này thì dân khổ quá,1,OFFENSIVE,data\processed\test_processed.csv,e29bf1a2d7e15d28fb3d498d8a93d705
2,vihsd_test_2,the nay ma chi phat gay roi trat tu cong cong ...,the nay ma chi phat gay roi trat tu cong cong ...,0,CLEAN,data\processed\test_processed.csv,2214bb090e01241ceef464d54a1ecb93
3,vihsd_test_3,Mua cho em hộp bcs mĩ sài cho oai :)),mua cho em hộp bcs mĩ sài cho oai :)),0,CLEAN,data\processed\test_processed.csv,1ff5898c13f93a8b153119d76e7876ea
4,vihsd_test_4,coin card :3,coin card :3,1,OFFENSIVE,data\processed\test_processed.csv,c10deb0b08b1d43c6ad344a6fbd2be48


In [5]:
vihsd_audit = pd.DataFrame([
    {"metric": "n_rows", "value": len(vihsd_test_std)},
    {"metric": "n_unique_text_norm", "value": vihsd_test_std["text_norm"].nunique()},
    {"metric": "n_duplicate_text_norm", "value": int(vihsd_test_std["text_norm"].duplicated().sum())},
    {"metric": "n_missing_text", "value": int((vihsd_test_std["text_norm"] == "").sum())},
])

label_audit = (
    vihsd_test_std["label_name"]
    .value_counts(dropna=False)
    .rename_axis("label_name")
    .reset_index(name="n_rows")
)

vihsd_audit.to_csv(RESULTS_DIR / "data_audit_vihsd.csv", index=False)
label_audit.to_csv(RESULTS_DIR / "data_audit_vihsd_label_distribution.csv", index=False)

vihsd_audit

,metric,value
0,n_rows,6680
1,n_unique_text_norm,6575
2,n_duplicate_text_norm,105
3,n_missing_text,0


## ViHOS manifest

In [6]:
def file_info(component, split, path):
    path = Path(path)
    info = {
        "component": component,
        "split": split,
        "file_path": rel_path(path),
        "exists": path.exists(),
        "n_rows": None,
        "n_cols": None,
        "columns": None,
    }
    if path.exists():
        df = read_csv(path)
        info["n_rows"] = len(df)
        info["n_cols"] = len(df.columns)
        info["columns"] = json.dumps(list(df.columns), ensure_ascii=False)
    return info

span_files = {
    "train": SPAN_DIR / "train.csv",
    "dev": SPAN_DIR / "dev.csv",
    "test": VIHOS_TEST_DIR / "test.csv",
}

manifest_rows = [file_info("span_extraction", split, path) for split, path in span_files.items()]

for path in sorted(BIO_DIR.rglob("*.csv")):
    split = path.stem.split("_")[0].lower()
    component = "bio_" + path.parent.name.lower()
    manifest_rows.append(file_info(component, split, path))

vihos_manifest = pd.DataFrame(manifest_rows)
vihos_manifest.to_csv(RESULTS_DIR / "vihos_file_manifest.csv", index=False)

vihos_manifest

,component,split,file_path,exists,n_rows,n_cols,columns
0,span_extraction,train,data\vihos\repo\data\Span_Extraction_based_ver...,True,8844,3,"[""Unnamed: 0"", ""content"", ""index_spans""]"
1,span_extraction,dev,data\vihos\repo\data\Span_Extraction_based_ver...,True,1106,3,"[""Unnamed: 0"", ""content"", ""index_spans""]"
2,span_extraction,test,data\vihos\repo\data\Test_data\test.csv,True,1106,3,"[""Unnamed: 0"", ""content"", ""index_spans""]"
3,bio_syllable,dev,data\vihos\repo\data\Sequence_labeling_based_v...,True,14476,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
4,bio_syllable,test,data\vihos\repo\data\Sequence_labeling_based_v...,True,13916,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
5,bio_syllable,train,data\vihos\repo\data\Sequence_labeling_based_v...,True,112754,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
6,bio_word,dev,data\vihos\repo\data\Sequence_labeling_based_v...,True,13948,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
7,bio_word,test,data\vihos\repo\data\Sequence_labeling_based_v...,True,13424,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
8,bio_word,train,data\vihos\repo\data\Sequence_labeling_based_v...,True,108432,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."


## ViHOS span

In [7]:
def standardize_vihos_span(path, split):
    df = read_csv(path)
    if "content" not in df.columns:
        raise ValueError(f"Khong thay cot content trong {path}")

    span_col = "index_spans" if "index_spans" in df.columns else None
    if span_col is None and "span_ids" in df.columns:
        span_col = "span_ids"
    if span_col is None:
        raise ValueError(f"Khong thay cot index_spans/span_ids trong {path}")

    rows = []
    for i, row in df.iterrows():
        content = "" if pd.isna(row["content"]) else str(row["content"])
        indices = parse_index_spans(row[span_col])
        ranges = indices_to_ranges(indices, len(content))
        span_texts = get_span_texts(content, ranges)

        rows.append({
            "sample_id": f"vihos_{split}_{i}",
            "split": split,
            "content": content,
            "content_norm": normalize_text(content),
            "index_spans": json.dumps(indices, ensure_ascii=False),
            "span_ranges": json.dumps(ranges, ensure_ascii=False),
            "span_texts": json.dumps(span_texts, ensure_ascii=False),
            "span_text": " | ".join(span_texts),
            "has_toxic_span": bool(span_texts),
            "n_spans": len(span_texts),
            "n_toxic_chars": len(indices),
            "text_hash": text_hash(content),
            "source_file": rel_path(path),
        })

    return pd.DataFrame(rows)

vihos_span = {}
for split, path in span_files.items():
    vihos_span[split] = standardize_vihos_span(path, split)
    out_path = RESOURCES_DIR / f"vihos_span_{split}_standard.csv"
    vihos_span[split].to_csv(out_path, index=False)

vihos_span["train"].head()

,sample_id,split,content,content_norm,index_spans,span_ranges,span_texts,span_text,has_toxic_span,n_spans,n_toxic_chars,text_hash,source_file
0,vihos_train_0,train,Dừa lắm :)),dừa lắm :)),[],[],[],,False,0,0,3a1b42e68dae0610979c3b2076f696c4,data\vihos\repo\data\Span_Extraction_based_ver...
1,vihos_train_1,train,Bấp bênh vl thế,bấp bênh vl thế,"[9, 10]","[[9, 11]]","[""vl""]",vl,True,1,2,58a4e4475985c32bcda2d19ea6dc4f07,data\vihos\repo\data\Span_Extraction_based_ver...
2,vihos_train_2,train,Chắc cũng biết ko tồn tại đc bao lâu nữa nên c...,chắc cũng biết ko tồn tại đc bao lâu nữa nên c...,"[53, 54, 55]","[[53, 56]]","[""vét""]",vét,True,1,3,c0b8060bd32320d560ffffc9e24309b3,data\vihos\repo\data\Span_Extraction_based_ver...
3,vihos_train_3,train,Thấy chán ad page này kiến thức thì nông cản c...,thấy chán ad page này kiến thức thì nông cản c...,"[5, 6, 7, 8, 36, 37, 38, 39, 40, 41, 42, 43, 6...","[[5, 9], [36, 44], [62, 65], [80, 88], [103, 1...","[""chán"", ""nông cản"", ""sủa"", ""tiêu cực"", ""dốt"",...",chán | nông cản | sủa | tiêu cực | dốt | nát,True,6,29,427cad6bdf03b355c350a35b734eba7c,data\vihos\repo\data\Span_Extraction_based_ver...
4,vihos_train_4,train,Giang Giang Đỗ Thị Ngọc Hà trend mới kìa kìa,giang giang đỗ thị ngọc hà trend mới kìa kìa,[],[],[],,False,0,0,1847e25c70c571a1653d338a87c7a3eb,data\vihos\repo\data\Span_Extraction_based_ver...


In [8]:
audit_rows = []
for split, df in vihos_span.items():
    n_index = df["index_spans"].map(lambda x: len(json.loads(x))).gt(0).sum()
    n_span = df["has_toxic_span"].sum()
    n_lost = ((df["index_spans"].map(lambda x: len(json.loads(x))) > 0) & (~df["has_toxic_span"])).sum()
    audit_rows.append({
        "split": split,
        "n_rows": len(df),
        "n_rows_with_index_spans": int(n_index),
        "n_rows_with_span_text": int(n_span),
        "n_rows_index_not_empty_but_span_empty": int(n_lost),
        "avg_n_spans": float(df["n_spans"].mean()),
        "avg_n_toxic_chars": float(df["n_toxic_chars"].mean()),
    })

vihos_span_audit = pd.DataFrame(audit_rows)
vihos_span_audit.to_csv(RESULTS_DIR / "data_audit_vihos_span.csv", index=False)

vihos_span_audit

,split,n_rows,n_rows_with_index_spans,n_rows_with_span_text,n_rows_index_not_empty_but_span_empty,avg_n_spans,avg_n_toxic_chars
0,train,8844,4292,4292,0,1.141565,8.752827
1,dev,1106,537,537,0,1.204340,9.101266
2,test,1106,531,531,0,1.144665,8.373418


## BIO manifest

In [9]:
bio_rows = []
for path in sorted(BIO_DIR.rglob("*.csv")):
    df = read_csv(path)
    bio_rows.append({
        "level": path.parent.name,
        "split": path.stem.split("_")[0].lower(),
        "file_path": rel_path(path),
        "n_rows": len(df),
        "n_cols": len(df.columns),
        "columns": json.dumps(list(df.columns), ensure_ascii=False),
    })

vihos_bio_manifest = pd.DataFrame(bio_rows)
vihos_bio_manifest.to_csv(RESULTS_DIR / "vihos_bio_manifest.csv", index=False)

vihos_bio_manifest

,level,split,file_path,n_rows,n_cols,columns
0,Syllable,dev,data\vihos\repo\data\Sequence_labeling_based_v...,14476,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
1,Syllable,test,data\vihos\repo\data\Sequence_labeling_based_v...,13916,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
2,Syllable,train,data\vihos\repo\data\Sequence_labeling_based_v...,112754,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
3,Word,dev,data\vihos\repo\data\Sequence_labeling_based_v...,13948,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
4,Word,test,data\vihos\repo\data\Sequence_labeling_based_v...,13424,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."
5,Word,train,data\vihos\repo\data\Sequence_labeling_based_v...,108432,5,"[""Unnamed: 0"", ""index"", ""Word"", ""Tag"", ""senten..."


## Rò rỉ dữ liệu

In [10]:
leakage_rows = []
for split, df in vihos_span.items():
    overlap_hash = set(vihsd_test_std["text_hash"]).intersection(set(df["text_hash"]))
    leakage_rows.append({
        "source_a": "vihsd_test",
        "source_b": f"vihos_{split}",
        "n_a": len(vihsd_test_std),
        "n_b": len(df),
        "n_overlap": len(overlap_hash),
        "overlap_rate_a": len(overlap_hash) / len(vihsd_test_std),
        "overlap_rate_b": len(overlap_hash) / len(df),
    })

leakage_report = pd.DataFrame(leakage_rows)
leakage_report.to_csv(RESULTS_DIR / "leakage_overlap_report.csv", index=False)

leakage_report

,source_a,source_b,n_a,n_b,n_overlap,overlap_rate_a,overlap_rate_b
0,vihsd_test,vihos_train,6680,8844,1067,0.159731,0.120647
1,vihsd_test,vihos_dev,6680,1106,150,0.022455,0.135624
2,vihsd_test,vihos_test,6680,1106,141,0.021108,0.127486


## Từ/cụm từ độc hại

In [11]:
def clean_phrase(phrase):
    phrase = normalize_text(phrase)
    phrase = re.sub(r"\s+", " ", phrase).strip()
    return phrase

phrases = []
for raw in vihos_span["train"]["span_texts"]:
    for phrase in json.loads(raw):
        phrase = clean_phrase(phrase)
        if len(phrase) < 2:
            continue
        if not re.search(r"[\wÀ-ỹ]", phrase, flags=re.UNICODE):
            continue
        phrases.append(phrase)

toxic_phrases = (
    pd.Series(phrases, name="phrase_norm")
    .value_counts()
    .rename_axis("phrase_norm")
    .reset_index(name="count")
)

toxic_phrases["phrase_len"] = toxic_phrases["phrase_norm"].str.len()
toxic_phrases["n_words"] = toxic_phrases["phrase_norm"].str.split().str.len()
toxic_phrases["source_split"] = "train"

toxic_phrases_out = RESOURCES_DIR / "toxic_phrases_candidates_train.csv"
toxic_phrases.to_csv(toxic_phrases_out, index=False)

toxic_phrases.head(20)

,phrase_norm,count,phrase_len,n_words,source_split
0,nó,459,2,1,train
1,đéo,255,3,1,train
2,thằng,214,5,1,train
3,chửi,178,4,1,train
4,mày,147,3,1,train
5,con,142,3,1,train
6,ngu,138,3,1,train
7,tao,135,3,1,train
8,đm,118,2,1,train
9,vl,107,2,1,train


## Độ phủ trên ViHSD test

In [12]:
phrases_for_match = toxic_phrases["phrase_norm"].tolist()
phrases_for_match = sorted(phrases_for_match, key=len, reverse=True)


def first_phrase_hit(text):
    text = normalize_text(text)
    for phrase in phrases_for_match:
        if phrase in text:
            return phrase
    return None

coverage_df = vihsd_test_std[["sample_id", "label_id", "label_name", "text_norm"]].copy()
coverage_df["matched_phrase"] = coverage_df["text_norm"].map(first_phrase_hit)
coverage_df["has_phrase_hit"] = coverage_df["matched_phrase"].notna()

overall = pd.DataFrame([{
    "group": "ALL",
    "n_rows": len(coverage_df),
    "n_hit": int(coverage_df["has_phrase_hit"].sum()),
    "coverage_rate": float(coverage_df["has_phrase_hit"].mean()),
}])

by_label = (
    coverage_df
    .groupby("label_name", dropna=False)["has_phrase_hit"]
    .agg(n_rows="count", n_hit="sum", coverage_rate="mean")
    .reset_index()
    .rename(columns={"label_name": "group"})
)

coverage_report = pd.concat([overall, by_label], ignore_index=True)
coverage_report.to_csv(RESULTS_DIR / "vihos_phrase_coverage_on_vihsd_test.csv", index=False)

coverage_examples = coverage_df[coverage_df["has_phrase_hit"]].head(100)
coverage_examples.to_csv(RESULTS_DIR / "vihos_phrase_coverage_examples.csv", index=False)

coverage_report

,group,n_rows,n_hit,coverage_rate
0,ALL,6680,5391,0.807036
1,CLEAN,5548,4269,0.769466
2,HATE,688,684,0.994186
3,OFFENSIVE,444,438,0.986486


## Trạng thái artifact

In [13]:
artifact_paths = {
    "vihsd_test_standard": vihsd_test_out,
    "vihos_span_train_standard": RESOURCES_DIR / "vihos_span_train_standard.csv",
    "vihos_span_dev_standard": RESOURCES_DIR / "vihos_span_dev_standard.csv",
    "vihos_span_test_standard": RESOURCES_DIR / "vihos_span_test_standard.csv",
    "toxic_phrases_candidates_train": toxic_phrases_out,
    "extension_data_contract": RESOURCES_DIR / "extension_data_contract.json",
    "data_audit_vihsd": RESULTS_DIR / "data_audit_vihsd.csv",
    "data_audit_vihos_span": RESULTS_DIR / "data_audit_vihos_span.csv",
    "leakage_overlap_report": RESULTS_DIR / "leakage_overlap_report.csv",
}

artifact_status = pd.DataFrame([
    {
        "artifact": name,
        "path": rel_path(path),
        "exists": Path(path).exists(),
        "size_bytes": Path(path).stat().st_size if Path(path).exists() else 0,
    }
    for name, path in artifact_paths.items()
])

artifact_status.to_csv(RESULTS_DIR / "extension_artifact_status.csv", index=False)
artifact_status

,artifact,path,exists,size_bytes
0,vihsd_test_standard,outputs\resources\vihsd_test_clean_standard.csv,True,1458580
1,vihos_span_train_standard,outputs\resources\vihos_span_train_standard.csv,True,3157944
2,vihos_span_dev_standard,outputs\resources\vihos_span_dev_standard.csv,True,394140
3,vihos_span_test_standard,outputs\resources\vihos_span_test_standard.csv,True,363876
4,toxic_phrases_candidates_train,outputs\resources\toxic_phrases_candidates_tra...,True,137449
5,extension_data_contract,outputs\resources\extension_data_contract.json,False,0
6,data_audit_vihsd,outputs\results\data_audit_vihsd.csv,True,97
7,data_audit_vihos_span,outputs\results\data_audit_vihos_span.csv,True,303
8,leakage_overlap_report,outputs\results\leakage_overlap_report.csv,True,298


## Hợp đồng dữ liệu

In [14]:
contract = {
    "name": "extension_data_contract",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "inputs": {
        "vihsd_test_processed": rel_path(vihsd_test_path),
        "vihos_span_train": rel_path(span_files["train"]),
        "vihos_span_dev": rel_path(span_files["dev"]),
        "vihos_span_test": rel_path(span_files["test"]),
        "vihos_bio_dir": rel_path(BIO_DIR),
    },
    "resources": {
        "vihsd_test_clean_standard": rel_path(vihsd_test_out),
        "vihos_span_train_standard": rel_path(RESOURCES_DIR / "vihos_span_train_standard.csv"),
        "vihos_span_dev_standard": rel_path(RESOURCES_DIR / "vihos_span_dev_standard.csv"),
        "vihos_span_test_standard": rel_path(RESOURCES_DIR / "vihos_span_test_standard.csv"),
        "toxic_phrases_candidates_train": rel_path(toxic_phrases_out),
    },
    "results": {
        "extension_artifact_status": rel_path(RESULTS_DIR / "extension_artifact_status.csv"),
        "data_audit_vihsd": rel_path(RESULTS_DIR / "data_audit_vihsd.csv"),
        "vihos_file_manifest": rel_path(RESULTS_DIR / "vihos_file_manifest.csv"),
        "data_audit_vihos_span": rel_path(RESULTS_DIR / "data_audit_vihos_span.csv"),
        "vihos_bio_manifest": rel_path(RESULTS_DIR / "vihos_bio_manifest.csv"),
        "leakage_overlap_report": rel_path(RESULTS_DIR / "leakage_overlap_report.csv"),
        "vihos_phrase_coverage_on_vihsd_test": rel_path(RESULTS_DIR / "vihos_phrase_coverage_on_vihsd_test.csv"),
    },
    "schema": {
        "vihsd_test_clean_standard": list(vihsd_test_std.columns),
        "vihos_span_standard": list(vihos_span["train"].columns),
        "toxic_phrases_candidates_train": list(toxic_phrases.columns),
    },
}

contract_path = RESOURCES_DIR / "extension_data_contract.json"
with open(contract_path, "w", encoding="utf-8") as f:
    json.dump(contract, f, ensure_ascii=False, indent=2)

print("saved:", contract_path)

saved: d:\Thư bae ngốc ngếch\outputs\resources\extension_data_contract.json


## Tổng kết

In [15]:
summary = pd.DataFrame([
    {"item": "ViHSD test", "value": len(vihsd_test_std)},
    {"item": "ViHOS train", "value": len(vihos_span["train"])},
    {"item": "ViHOS dev", "value": len(vihos_span["dev"])},
    {"item": "ViHOS test", "value": len(vihos_span["test"])},
    {"item": "Toxic phrases", "value": len(toxic_phrases)},
    {"item": "Coverage ALL", "value": coverage_report.loc[coverage_report["group"] == "ALL", "coverage_rate"].iloc[0]},
])

summary

,item,value
0,ViHSD test,6680.000000
1,ViHOS train,8844.000000
2,ViHOS dev,1106.000000
3,ViHOS test,1106.000000
4,Toxic phrases,4426.000000
5,Coverage ALL,0.807036
